In [11]:
import Gmsh: gmsh

# Initializing Gmsh
gmsh.initialize()

gmsh.model.add("PlateHole_Q4")

# --- Parameters (Converted to Meters to match your L=1 scale) ---
L = 1.0      # 1000 mm
H = 0.4      # 400 mm
R = 0.075    # Radius 75 mm
Xc = 0.5     # Center X (500 mm)
Yc = 0.2     # Center Y (200 mm)

# Mesh sizes
h_coarse = 0.008  # Coarse mesh for far boundaries
h_fine   = 0.005 # Fine mesh for the hole (CRITICAL for stress concentration)

# --- 1. Define the Outer Rectangle ---
p1 = gmsh.model.geo.addPoint(0, 0, 0, h_coarse)
p2 = gmsh.model.geo.addPoint(L, 0, 0, h_coarse)
p3 = gmsh.model.geo.addPoint(L, H, 0, h_coarse)
p4 = gmsh.model.geo.addPoint(0, H, 0, h_coarse)

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)

# Outer Loop
cl_rect = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])

# --- 2. Define the Inner Circle ---
# We create 5 points: Center + 4 points on the circumference (Top, Right, Bottom, Left)
# This method (Circle Arcs) is more robust for meshing than a single full circle.

pc = gmsh.model.geo.addPoint(Xc, Yc, 0, h_fine) # Center
p_r = gmsh.model.geo.addPoint(Xc + R, Yc, 0, h_fine)
p_t = gmsh.model.geo.addPoint(Xc, Yc + R, 0, h_fine)
p_l = gmsh.model.geo.addPoint(Xc - R, Yc, 0, h_fine)
p_b = gmsh.model.geo.addPoint(Xc, Yc - R, 0, h_fine)

# Create 4 arcs to form the circle
c1 = gmsh.model.geo.addCircleArc(p_r, pc, p_t)
c2 = gmsh.model.geo.addCircleArc(p_t, pc, p_l)
c3 = gmsh.model.geo.addCircleArc(p_l, pc, p_b)
c4 = gmsh.model.geo.addCircleArc(p_b, pc, p_r)

# Inner Loop
cl_hole = gmsh.model.geo.addCurveLoop([c1, c2, c3, c4])

# --- 3. Create Surface with Hole ---
# The first loop is the boundary, subsequent loops are holes
s = gmsh.model.geo.addPlaneSurface([cl_rect, cl_hole])

# --- 4. Physical Groups (Boundary Conditions) ---
# Domain
gmsh.model.addPhysicalGroup(2, [s], 1, "Plate")

# Boundaries
gmsh.model.addPhysicalGroup(1, [l4], 2, "FixedLeft")  # Left Edge
gmsh.model.addPhysicalGroup(1, [l2], 3, "LoadRight")  # Right Edge
gmsh.model.addPhysicalGroup(1, [c1, c2, c3, c4], 4, "HoleEdge") # Hole boundary

# Synchronize and Mesh
gmsh.model.geo.synchronize()
gmsh.model.mesh.generate(2)

# Write mesh to file
gmsh.write("PlateHole_Q4.msh")

# Launch GUI to see the result
gmsh.fltk.run()
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 60%] Meshing curve 5 (Circle)
Info    : [ 70%] Meshing curve 6 (Circle)
Info    : [ 80%] Meshing curve 7 (Circle)
Info    : [ 90%] Meshing curve 8 (Circle)
Info    : Done meshing 1D (Wall 0.0022459s, CPU 0s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.31148s, CPU 0.296875s)
Info    : 9118 nodes 18243 elements
Info    : Writing 'PlateHole_Q4.msh'...
Info    : Done writing 'PlateHole_Q4.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : Windows64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration DomHex Eigen[contrib] Fltk GMP Gmm[contr

In [12]:
using Gridap
using GridapGmsh

In [13]:
# 1. Read the Mesh
model = GmshDiscreteModel("PlateHole_Q4.msh")

Info    : Reading 'PlateHole_Q4.msh'...
Info    : 18 entities
Info    : 9117 nodes
Info    : 17984 elements
Info    : Done reading 'PlateHole_Q4.msh'


UnstructuredDiscreteModel()

In [14]:
# 2. Material Parameters (Steel)
const E = 210000.0  # MPa
const ν = 0.3       # Poisson's ratio
const thickness = 2.0 # mm

2.0

In [15]:
# Lamé parameters for Plane Stress
const E = 210000.0                              # N/mm^2
const ν = 0.3                                   # Poissons Ratio

const μ = E/(2*(1+ν))
const λ = (E * ν) / ((1 + ν) * (1 - 2 * ν))
const λ_ps = (2 * λ * μ) / (λ + 2 * μ)          # Plane stress correction

g = VectorValue(100.0, 0.0)                # Force

σ(ε) = λ_ps*tr(ε)*one(ε) + 2*μ*ε

σ (generic function with 1 method)

In [16]:
# 3. Define FE Spaces
order = 1

reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V0 = TestFESpace(model,reffe;
    conformity=:H1,
    dirichlet_tags=["FixedLeft"],
    dirichlet_masks=[(true,true)])
  
g1 = VectorValue(0.0,0.0)
U = TrialFESpace(V0,[g1])

TrialFESpace()

In [17]:
# 4. Numerical Integration
degree = 2*order
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)
Γ_load  = BoundaryTriangulation(model, tags = "LoadRight")
dΓ_N = Measure(Γ_load,degree)

GenericMeasure()

In [18]:
# 5. Define Weak Form

# Internal Work (Stiffness)
a(u,v) = ∫( ε(v) ⊙ (σ∘ε(u)) )*dΩ 

# External Work (Load)
l(v) = ∫(v⋅g)*dΓ_N

l (generic function with 1 method)

In [19]:
# 6. Solve
op = AffineFEOperator(a,l,U,V0)
uh = solve(op)

SingleFieldFEFunction():
 num_cells: 17788
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 594834959736531065

In [20]:
# 7. Post-Process
writevtk(Ω,"results_Q4",
    cellfields=[
        "Displacement"=>uh,
        "Strain"=>ε(uh),
        "Stress"=>σ∘ε(uh)]
        )

(["results_Q4.vtu"],)